# LangChain + Sarvam RAG

A Retrieval-Augmented Generation pipeline that uses LangChain for vector search
and Sarvam AI `sarvam-m` for answer generation.

## Pipeline

```
User query
        |
        v
retrieve_documents()    <- LangChain FAISS + HuggingFaceEmbeddings
        |  query embedded with paraphrase-multilingual-mpnet-base-v2
        |  vector_store.as_retriever().invoke(query)
        v
generate_answer()       <- Sarvam sarvam-m (chat.completions)
        |  context: page_content from retrieved Documents
        v
Answer
```

In [ ]:
%pip install sarvamai>=0.1.26 langchain>=0.3.0 langchain-community>=0.3.0 sentence-transformers>=2.2.2 faiss-cpu>=1.7.4 python-dotenv>=1.0.0

## Setup and API Key

Load environment variables and initialise the Sarvam AI client.
Set `SARVAM_API_KEY` in a `.env` file copied from `.env.example`.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
print("Setup complete.")

## Sample Documents

Create two in-memory LangChain `Document` objects.
Replace these with your own data loaded from files, databases, or APIs.

In [ ]:
SAMPLE_DOCS: list[Document] = [
    Document(
        page_content=(
            "Sarvam AI is an Indian AI company building large language models and APIs "
            "optimised for Indian languages. Its offerings include speech-to-text, "
            "text-to-speech, translation, and chat completion APIs that support languages "
            "such as Hindi, Tamil, Telugu, Kannada, Bengali, and more."
        ),
        metadata={"source": "doc-1", "topic": "sarvam-ai"},
    ),
    Document(
        page_content=(
            "LangChain is an open-source framework for building applications powered by "
            "large language models. It provides abstractions for document loading, text "
            "splitting, embedding, vector storage, and retrieval chains. The FAISS "
            "integration allows fast in-memory similarity search over embedded documents."
        ),
        metadata={"source": "doc-2", "topic": "langchain"},
    ),
]

print(f"Created {len(SAMPLE_DOCS)} sample documents.")
for doc in SAMPLE_DOCS:
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"  [{doc.metadata['source']}] {preview}...")

## Build FAISS Vector Store

Index the sample documents using LangChain `FAISS.from_documents`.
The `HuggingFaceEmbeddings` wrapper calls the sentence-transformers model locally.

Model: `paraphrase-multilingual-mpnet-base-v2` (~420 MB first download, cached after that)

In [ ]:
def create_vector_store(
    documents: list[Document],
    model_name: str = EMBEDDING_MODEL,
) -> FAISS:
    """Build a LangChain FAISS vector store from a list of Documents.

    Args:
        documents: List of LangChain Document objects to index.
        model_name: HuggingFace model name used for sentence-transformers embeddings.

    Returns:
        Populated FAISS vector store ready for similarity search.
    """
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return FAISS.from_documents(documents, embeddings)

In [ ]:
vector_store = create_vector_store(SAMPLE_DOCS)

print(f"Vector store built.")
print(f"Embedding model  : {EMBEDDING_MODEL}")
print(f"Documents indexed: {vector_store.index.ntotal}")

## Retrieve Documents

Use LangChain's retriever interface to find the most relevant documents for a query.
The retriever works with any LangChain-compatible vector store, not just FAISS.

In [ ]:
def retrieve_documents(
    query: str,
    vector_store: FAISS,
    top_k: int = 2,
) -> list[Document]:
    """Retrieve the top-k most relevant documents for a query.

    Uses LangChain's standard retriever interface so this function works with
    any LangChain-compatible vector store, not just FAISS.

    Args:
        query: User query string.
        vector_store: Populated LangChain FAISS vector store.
        top_k: Number of documents to return.

    Returns:
        List of LangChain Document objects ordered by relevance.
    """
    retriever = vector_store.as_retriever(search_kwargs={"k": top_k})
    return retriever.invoke(query)

## Generate Answer

Pass the retrieved document `page_content` as context to Sarvam `sarvam-m`.
No custom LangChain LLM adapter is required — the Sarvam client is called directly.

In [ ]:
def generate_answer(
    query: str,
    retrieved_docs: list[Document],
    sarvam_client: SarvamAI,
) -> str:
    """Generate an answer using Sarvam sarvam-m with LangChain-retrieved context.

    Args:
        query: Original user query.
        retrieved_docs: List of Documents returned by the LangChain retriever.
        sarvam_client: Initialised SarvamAI client.

    Returns:
        Answer string from sarvam-m.
    """
    context = "\n\n".join(doc.page_content for doc in retrieved_docs)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the question using only the "
                "provided context. If the answer is not in the context, say so."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}",
        },
    ]

    response = sarvam_client.chat.completions(messages=messages, model="sarvam-m")
    return response.choices[0].message.content

## Full RAG Pipeline

`run_rag_pipeline` chains LangChain retrieval and Sarvam generation end-to-end.

In [ ]:
def run_rag_pipeline(
    query: str,
    vector_store: FAISS,
    sarvam_client: SarvamAI,
    top_k: int = 2,
) -> dict[str, Any]:
    """Run the full LangChain + Sarvam RAG pipeline.

    Steps:
        1. Retrieve top-k relevant documents via LangChain FAISS retriever.
        2. Generate an answer with Sarvam sarvam-m using retrieved context.

    Args:
        query: User query string.
        vector_store: Populated LangChain FAISS vector store.
        sarvam_client: Initialised SarvamAI client.
        top_k: Number of documents to retrieve.

    Returns:
        Dict with keys: query, retrieved_sources, answer.
    """
    retrieved = retrieve_documents(query, vector_store, top_k)
    answer = generate_answer(query, retrieved, sarvam_client)

    return {
        "query": query,
        "retrieved_sources": [doc.metadata.get("source", "unknown") for doc in retrieved],
        "answer": answer,
    }


# Demo
result = run_rag_pipeline(
    query="What is Sarvam AI?",
    vector_store=vector_store,
    sarvam_client=client,
)

print("Query            :", result["query"])
print("Retrieved sources:", result["retrieved_sources"])
print()
print("Answer:")
print(result["answer"])

## Error Reference and Resources

### Common Errors

| Error | Cause | Fix |
|:------|:------|:----|
| `RuntimeError: SARVAM_API_KEY is not set` | Missing `.env` | Copy `.env.example` to `.env` and add your key |
| `ModuleNotFoundError: faiss` | FAISS not installed | `pip install faiss-cpu` |
| `ModuleNotFoundError: langchain_community` | Package not installed | `pip install langchain-community` |
| `sarvamai.APIStatusError: 401` | Invalid API key | Verify key at dashboard.sarvam.ai |
| `sarvamai.APIStatusError: 429` | Rate limit exceeded | Add a short delay between requests |

### Resources

- [Sarvam AI Documentation](https://docs.sarvam.ai/)
- [Sarvam API Dashboard](https://dashboard.sarvam.ai/)
- [LangChain FAISS](https://python.langchain.com/docs/integrations/vectorstores/faiss/)
- [LangChain HuggingFaceEmbeddings](https://python.langchain.com/docs/integrations/text_embedding/huggingfacehub/)
- [paraphrase-multilingual-mpnet-base-v2](https://huggingface.co/sentence-transformers/paraphrase-multilingual-mpnet-base-v2)